# Feature Engineering — Caso 3: Compensaciones por Posible Fraude

**Input:** `data/Dataset_caso_3.xlsx` (150 casos originales, pestaña `Caso3_Compensaciones`).

**Objetivo:** derivar las variables que alimentan el motor de reglas heurísticas (step 03) y el pipeline LangGraph (step 06). La lógica del pipeline es:

```
datos base → FeatureEngineer → RuleEngine → decisión
```

**Outputs:**

- `data/casos_con_features.parquet` — dataset completo con las 16 features nuevas
- PostgreSQL `casos` — columnas derivadas actualizadas para los 150 casos

## Outline

1. Setup y carga
2. Variables de riesgo financiero (ratios)
3. Variables de inconsistencia (flags booleanos)
4. Variables de análisis de texto (meta-features)
5. Variables geográficas agregadas
6. Variables derivadas del EDA
7. Validación del dataset enriquecido
8. Persistencia: parquet + PostgreSQL

## 1. Setup y carga

Cargamos los 150 casos originales del Excel. Los percentiles del EDA (step 01) se recalculan aquí para usar como thresholds en las features que los necesitan.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

CANDIDATAS = [Path('data/Dataset_caso_3.xlsx'), Path('../data/Dataset_caso_3.xlsx')]
EXCEL_PATH = next((p for p in CANDIDATAS if p.exists()), None)
assert EXCEL_PATH is not None, 'No se encontró el dataset'

df = pd.read_excel(EXCEL_PATH, sheet_name='Caso3_Compensaciones', header=1)
print(f'Casos cargados: {len(df)}')
df.head(3)

Casos cargados: 150


,caso_id,usuario_id,antiguedad_usuario_dias,ciudad,vertical,restaurante,valor_orden_mxn,compensacion_solicitada_mxn,num_compensaciones_90d,monto_compensado_90d_mxn,entrega_confirmada_gps,tiempo_entrega_real_min,flags_fraude_previos,motivo_reclamo,descripcion_reclamo,recomendacion_agente
0,COMP-0001,USR-11567,1590,CDMX,Comida,La Cocina de Doña Rosa,668.79,423.76,2,103.53,NO confirmada,56,0,Producto incorrecto,"Llegó comida diferente, creo que confundieron ...",PENDIENTE
1,COMP-0002,maximosabor,229,Bogotá,Mercado,El Rincón Colombiano,166.56,153.91,2,334.39,Parcial,60,1,Producto incorrecto,Me mandaron una hamburguesa con queso y tengo ...,PENDIENTE
2,COMP-0003,USR-13864,200,Buenos Aires,Mercado,La Cocina de Doña Rosa,226.34,203.15,2,691.29,NO confirmada,53,1,Producto incompleto,No incluyeron las salsas ni los cubiertos.,PENDIENTE


In [2]:
# Thresholds del EDA (percentiles del dataset original)
P90_TIEMPO_ENTREGA = df['tiempo_entrega_real_min'].quantile(0.90)
P95_COMPENSACION = df['compensacion_solicitada_mxn'].quantile(0.95)
P95_NUM_COMPS = df['num_compensaciones_90d'].quantile(0.95)
P90_NUM_COMPS = df['num_compensaciones_90d'].quantile(0.90)

print(f'p90 tiempo_entrega_real_min: {P90_TIEMPO_ENTREGA:.0f} min')
print(f'p95 compensacion_solicitada: {P95_COMPENSACION:.2f} MXN')
print(f'p95 num_compensaciones_90d:  {P95_NUM_COMPS:.0f}')
print(f'p90 num_compensaciones_90d:  {P90_NUM_COMPS:.0f}')

p90 tiempo_entrega_real_min: 96 min
p95 compensacion_solicitada: 528.28 MXN
p95 num_compensaciones_90d:  10
p90 num_compensaciones_90d:  9


## 2. Variables de riesgo financiero (ratios)

Detectan **abuso de políticas** y **granjas de cuentas**: el patrón de pedir mucho dinero respecto al valor de la orden, o acumular compensaciones muy rápido.

| Feature | Fórmula | Qué detecta |
|---|---|---|
| `comp_ratio` | compensación / valor_orden | Pide más de lo que pagó (ratio > 2.0 = absurdo) |
| `burn_rate` | monto_compensado_90d / antigüedad_días | Cuentas nuevas "ordeñando" el sistema |
| `freq_densidad` | comps_90d / min(antigüedad, 90) | Densidad de reclamos: 3 en 7 días ≠ 3 en 4 años |

In [3]:
def build_financial_features(df: pd.DataFrame) -> pd.DataFrame:
    """Deriva ratios de riesgo financiero.

    Args:
        df: DataFrame con las columnas originales del dataset.

    Returns:
        DataFrame con comp_ratio, burn_rate y freq_densidad añadidas.
    """
    df = df.copy()
    df['comp_ratio'] = (
        df['compensacion_solicitada_mxn'] / df['valor_orden_mxn'].clip(lower=1)
    )
    df['burn_rate'] = (
        df['monto_compensado_90d_mxn'] / df['antiguedad_usuario_dias'].clip(lower=1)
    )
    df['freq_densidad'] = (
        df['num_compensaciones_90d'] / df['antiguedad_usuario_dias'].clip(upper=90).clip(lower=1)
    )
    return df

df = build_financial_features(df)
df[['caso_id', 'valor_orden_mxn', 'compensacion_solicitada_mxn',
    'comp_ratio', 'burn_rate', 'freq_densidad']].head()

,caso_id,valor_orden_mxn,compensacion_solicitada_mxn,comp_ratio,burn_rate,freq_densidad
0,COMP-0001,668.79,423.76,0.633622,0.065113,0.022222
1,COMP-0002,166.56,153.91,0.924051,1.460218,0.022222
2,COMP-0003,226.34,203.15,0.897544,3.456450,0.022222
3,COMP-0004,299.73,285.90,0.953858,0.086062,0.022222
4,COMP-0005,451.20,340.82,0.755363,0.007154,0.000000


### Lectura

- **`comp_ratio`**: la mediana del dataset ronda 0.8 (la compensación típica es ~80% del valor de la orden). Un ratio > 2.0 significa pedir el doble de lo pagado — señal de RECHAZAR.
- **`burn_rate`**: usuarios con años de antigüedad tienen burn_rate bajo aunque hayan acumulado compensaciones; una cuenta de 15 días con 2,000 MXN compensados tiene un burn_rate altísimo.
- **`freq_densidad`**: normaliza reclamos por la ventana real de exposición del usuario (máximo 90 días, que es la ventana del dato).

## 3. Variables de inconsistencia (flags booleanos)

Detectan **contradicciones lógicas** entre lo que el usuario afirma y lo que los datos duros muestran. Son el insumo del *filtro duro* del motor de reglas.

| Feature | Condición | Señal |
|---|---|---|
| `flag_inconsistencia_gps` | "Orden no llegó" pero GPS confirma entrega | Mentira directa |
| `flag_mentira_gps_alta` | Reclamo de producto + GPS ok + compensación > p95 | Fraude sofisticado |
| `flag_retraso_critico` | tiempo_entrega > p90 (96 min) | Queja de demora respaldada por datos |
| `flag_account_abuse` | antigüedad < 90d Y reclamos > p95 (10) | Cuenta nueva con frecuencia récord |
| `score_riesgo_previo` | flags×2 + comps_90d×0.5 | Riesgo histórico acumulado |

In [4]:
def build_inconsistency_features(df: pd.DataFrame) -> pd.DataFrame:
    """Deriva flags de inconsistencia lógica.

    Args:
        df: DataFrame con columnas originales.

    Returns:
        DataFrame con flags booleanos y score_riesgo_previo añadidos.
    """
    df = df.copy()

    # El usuario dice que no llegó, pero el GPS confirma (o confirma parcial)
    df['flag_inconsistencia_gps'] = (
        (df['motivo_reclamo'] == 'Orden no llegó')
        & (df['entrega_confirmada_gps'].isin(['SÍ - confirmada', 'Parcial']))
    )

    # Reclamo fino (producto) con GPS ok pero pide compensación alta
    df['flag_mentira_gps_alta'] = (
        (df['motivo_reclamo'].isin(['Producto incorrecto', 'Producto incompleto']))
        & (df['entrega_confirmada_gps'] == 'SÍ - confirmada')
        & (df['compensacion_solicitada_mxn'] > P95_COMPENSACION)
    )

    # Demora crítica respaldada por datos duros → auto-aprobar si el motivo es demora
    df['flag_retraso_critico'] = df['tiempo_entrega_real_min'] > P90_TIEMPO_ENTREGA

    # Cuenta nueva con frecuencia récord de reclamos
    df['flag_account_abuse'] = (
        (df['antiguedad_usuario_dias'] < 90)
        & (df['num_compensaciones_90d'] > P95_NUM_COMPS)
    )

    # Riesgo histórico acumulado
    df['score_riesgo_previo'] = (
        df['flags_fraude_previos'] * 2 + df['num_compensaciones_90d'] * 0.5
    )

    return df

df = build_inconsistency_features(df)
print('Distribución de flags:')
for col in ['flag_inconsistencia_gps', 'flag_mentira_gps_alta',
            'flag_retraso_critico', 'flag_account_abuse']:
    print(f'  {col}: {int(df[col].sum())} casos True')
print(f'\nscore_riesgo_previo: media={df["score_riesgo_previo"].mean():.2f}, '
      f'max={df["score_riesgo_previo"].max():.1f}')

Distribución de flags:
  flag_inconsistencia_gps: 9 casos True
  flag_mentira_gps_alta: 0 casos True
  flag_retraso_critico: 14 casos True
  flag_account_abuse: 7 casos True

score_riesgo_previo: media=3.86, max=13.5


### Lectura

- **`flag_inconsistencia_gps`** es la señal más limpia: contradicción verificable entre el reclamo y el dato de telemetría. RECHAZAR directo en las reglas.
- **`flag_retraso_critico`** funciona al revés: respalda reclamos legítimos por demora. Si el motivo es "Orden llegó tarde" y este flag es True → APROBAR candidato.
- **`score_riesgo_previo`** convierte el historial en un número comparable entre casos; su threshold se define en `thresholds.yaml` (step 03).

## 4. Variables de análisis de texto (meta-features)

Extraídas **sin gastar tokens de LLM**: solo forma del texto, no su contenido semántico.

| Feature | Lógica | Señal |
|---|---|---|
| `longitud_reclamo` | palabras en descripcion_reclamo | Extremos sospechosos: 3 palabras o párrafos elaborados |
| `flag_palabras_criticas` | regex de liability (alergia, intoxicado, policía…) | **ESCALAR forzoso** — seguridad de marca |

In [5]:
import re

PALABRAS_CRITICAS = re.compile(
    r'alergi|intoxic|polic[ií]a|sangre|insult|denunci|abogado|demanda|hospital|veneno',
    re.IGNORECASE,
)

def build_text_features(df: pd.DataFrame) -> pd.DataFrame:
    """Deriva meta-features del texto del reclamo.

    Args:
        df: DataFrame con columna descripcion_reclamo.

    Returns:
        DataFrame con longitud_reclamo y flag_palabras_criticas.
    """
    df = df.copy()
    texto = df['descripcion_reclamo'].fillna('')
    df['longitud_reclamo'] = texto.str.split().str.len()
    df['flag_palabras_criticas'] = texto.str.contains(PALABRAS_CRITICAS)
    return df

df = build_text_features(df)
print(f'longitud_reclamo: min={df["longitud_reclamo"].min()}, '
      f'media={df["longitud_reclamo"].mean():.1f}, max={df["longitud_reclamo"].max()}')
print(f'flag_palabras_criticas: {int(df["flag_palabras_criticas"].sum())} casos True')
df.loc[df['flag_palabras_criticas'], ['caso_id', 'descripcion_reclamo']].head()

longitud_reclamo: min=6, media=10.1, max=16
flag_palabras_criticas: 9 casos True


,caso_id,descripcion_reclamo
1,COMP-0002,Me mandaron una hamburguesa con queso y tengo ...
71,COMP-0072,Me mandaron una hamburguesa con queso y tengo ...
73,COMP-0074,Me mandaron una hamburguesa con queso y tengo ...
109,COMP-0110,Me mandaron una hamburguesa con queso y tengo ...
115,COMP-0116,Me mandaron una hamburguesa con queso y tengo ...


### Lectura

- **`flag_palabras_criticas`** tiene precedencia sobre TODO: si un reclamo menciona alergias, intoxicación o policía, el caso se ESCALA a un humano sin importar el score. Un falso negativo aquí cuesta mucho más que un falso positivo.
- **`longitud_reclamo`** alimenta el análisis del pipeline: textos muy cortos (poca evidencia) o muy largos (sobre-elaborados) pesan distinto en la decisión.

## 5. Variables geográficas agregadas

Tasas de reclamo relativas por ciudad y vertical. Se calculan sobre los 150 casos originales (los sintéticos heredan la tasa de su ciudad/vertical, no la alteran).

- `riesgo_ciudad = casos_en_ciudad / total_casos` (conteo relativo)
- `riesgo_vertical = casos_en_vertical / total_casos`

In [6]:
def build_geo_features(df: pd.DataFrame) -> pd.DataFrame:
    """Deriva tasas de reclamo por ciudad y vertical.

    Args:
        df: DataFrame con columnas ciudad y vertical.

    Returns:
        DataFrame con riesgo_ciudad y riesgo_vertical (0-1).
    """
    df = df.copy()
    total = len(df)
    df['riesgo_ciudad'] = df['ciudad'].map(df['ciudad'].value_counts() / total)
    df['riesgo_vertical'] = df['vertical'].map(df['vertical'].value_counts() / total)
    return df

df = build_geo_features(df)
df.groupby('ciudad')['riesgo_ciudad'].first().sort_values(ascending=False).head()

ciudad
CDMX            0.180000
Guadalajara     0.113333
Buenos Aires    0.086667
Lima            0.080000
Bogotá          0.073333
Name: riesgo_ciudad, dtype: float64

## 6. Variables derivadas del EDA

Capturan los insights descubiertos en el notebook `01_eda.ipynb`:

- **Paradoja GPS**: quien tiene GPS confirmada y AUN ASÍ reclama mucho, es el defraudador sofisticado (reclamos chicos y frecuentes).
- **Antigüedad baja + reclamos + flags**: la tríada de mayor riesgo del EDA.
- **Z-score del comp_ratio**: cuántas desviaciones estándar se desvía del promedio.

In [7]:
def build_eda_features(df: pd.DataFrame) -> pd.DataFrame:
    """Deriva features basadas en insights del EDA (step 01).

    Args:
        df: DataFrame con columnas originales y comp_ratio.

    Returns:
        DataFrame con gps_paradoja_score, sospecha_nuevo_recurrente, ratio_deviation.
    """
    df = df.copy()

    # GPS ok + reclama mucho = fraude sofisticado (score continuo 0-1)
    gps_ok = df['entrega_confirmada_gps'] == 'SÍ - confirmada'
    frecuencia_alta = df['num_compensaciones_90d'] > P90_NUM_COMPS
    df['gps_paradoja_score'] = (
        gps_ok.astype(float) * 0.5
        + frecuencia_alta.astype(float) * 0.3
        + (df['flags_fraude_previos'] > 0).astype(float) * 0.2
    )

    # Tríada de riesgo: usuario nuevo + reclamos + flags
    df['sospecha_nuevo_recurrente'] = (
        (df['antiguedad_usuario_dias'] < 90)
        & (df['num_compensaciones_90d'] >= 3)
        & (df['flags_fraude_previos'] >= 1)
    )

    # Z-score del comp_ratio
    media, std = df['comp_ratio'].mean(), df['comp_ratio'].std()
    df['ratio_deviation'] = (df['comp_ratio'] - media) / std if std > 0 else 0.0

    return df

df = build_eda_features(df)
print(f'gps_paradoja_score: media={df["gps_paradoja_score"].mean():.2f}, '
      f'max={df["gps_paradoja_score"].max():.2f}')
print(f'sospecha_nuevo_recurrente: {int(df["sospecha_nuevo_recurrente"].sum())} casos True')
print(f'ratio_deviation: min={df["ratio_deviation"].min():.2f}, '
      f'max={df["ratio_deviation"].max():.2f}')

gps_paradoja_score: media=0.26, max=1.00
sospecha_nuevo_recurrente: 33 casos True
ratio_deviation: min=-2.24, max=1.45


## 7. Validación del dataset enriquecido

Verificamos que:

- Las 16 features existen y no tienen nulos inesperados
- Los tipos son correctos (bool en flags, float en ratios)
- No hay infinitos por divisiones

In [8]:
FEATURES = [
    'comp_ratio', 'burn_rate', 'freq_densidad',
    'flag_inconsistencia_gps', 'flag_mentira_gps_alta', 'flag_retraso_critico',
    'flag_account_abuse', 'score_riesgo_previo',
    'longitud_reclamo', 'flag_palabras_criticas',
    'riesgo_ciudad', 'riesgo_vertical',
    'gps_paradoja_score', 'sospecha_nuevo_recurrente', 'ratio_deviation',
    # placeholder del pipeline (se llena con LLM en step 06)
    'score_texto',
]

df['score_texto'] = np.nan  # lo completa el nodo llm_classify del pipeline

faltantes = [f for f in FEATURES if f not in df.columns]
assert not faltantes, f'Features faltantes: {faltantes}'

print(f'Total features: {len(FEATURES)}')
print(f'Nulos por feature (score_texto es NaN a propósito):')
nulos = df[FEATURES].isna().sum()
print(nulos.to_string())
assert not np.isinf(df[['comp_ratio', 'burn_rate', 'freq_densidad']]).any().any(), 'Hay infinitos'
print('\n[OK] Validación superada: 16 features, sin infinitos.')

Total features: 16
Nulos por feature (score_texto es NaN a propósito):
comp_ratio                     0
burn_rate                      0
freq_densidad                  0
flag_inconsistencia_gps        0
flag_mentira_gps_alta          0
flag_retraso_critico           0
flag_account_abuse             0
score_riesgo_previo            0
longitud_reclamo               0
flag_palabras_criticas         0
riesgo_ciudad                  0
riesgo_vertical                0
gps_paradoja_score             0
sospecha_nuevo_recurrente      0
ratio_deviation                0
score_texto                  150

[OK] Validación superada: 16 features, sin infinitos.


## 8. Persistencia: parquet + PostgreSQL

1. **Parquet**: dataset completo (originales + features) para el pipeline.
2. **PostgreSQL**: `UPDATE` de las columnas derivadas en la tabla `casos`, para que el pipeline lea de la DB y no del Excel.

In [9]:
DATA_DIR = Path('data') if Path('data').exists() else Path('../data')
PARQUET_PATH = DATA_DIR / 'casos_con_features.parquet'

df.to_parquet(PARQUET_PATH, index=False)
print(f'[OK] Parquet guardado: {PARQUET_PATH} ({len(df)} filas, {len(df.columns)} columnas)')

[OK] Parquet guardado: ../data/casos_con_features.parquet (150 filas, 32 columnas)


In [10]:
import psycopg2

DB_CONFIG = {
    'host': 'localhost', 'port': 5432, 'dbname': 'rappi_cases',
    'user': 'rappi', 'password': 'rappi_pass',
}

MAPA_DB = {
    'comp_ratio': 'comp_ratio',
    'burn_rate': 'burn_rate',
    'freq_densidad': 'freq_densidad',
    'flag_inconsistencia_gps': 'flag_inconsistencia_gps',
    'flag_mentira_gps_alta': 'flag_mentira_gps_alta',
    'flag_retraso_critico': 'flag_retraso_critico',
    'flag_account_abuse': 'flag_account_abuse',
    'score_riesgo_previo': 'score_riesgo_previo',
    'longitud_reclamo': 'longitud_reclamo',
    'flag_palabras_criticas': 'flag_palabras_criticas',
    'riesgo_ciudad': 'riesgo_ciudad',
    'riesgo_vertical': 'riesgo_vertical',
    'gps_paradoja_score': 'gps_paradoja_score',
    'sospecha_nuevo_recurrente': 'sospecha_nuevo_recurrente',
    'ratio_deviation': 'ratio_deviation',
}

conn = psycopg2.connect(**DB_CONFIG)
actualizados = 0
try:
    with conn.cursor() as cur:
        for _, fila in df.iterrows():
            sets = ', '.join(f'{col} = %s' for col in MAPA_DB.values())
            valores = [
                (bool(fila[c]) if isinstance(fila[c], (bool, np.bool_)) else
                 float(fila[c]) if isinstance(fila[c], (int, float, np.floating, np.integer)) and not pd.isna(fila[c]) else None)
                for c in MAPA_DB.keys()
            ]
            cur.execute(
                f'UPDATE casos SET {sets} WHERE caso_id = %s AND es_sintetico = FALSE',
                valores + [fila['caso_id']],
            )
            actualizados += cur.rowcount
    conn.commit()
finally:
    conn.close()

print(f'[OK] PostgreSQL actualizado: {actualizados} casos')

[OK] PostgreSQL actualizado: 150 casos


In [11]:
# Verificación final desde la DB
conn = psycopg2.connect(**DB_CONFIG)
check = pd.read_sql(
    """SELECT COUNT(*) AS total,
              COUNT(comp_ratio) AS con_comp_ratio,
              COUNT(gps_paradoja_score) AS con_paradoja
       FROM casos WHERE es_sintetico = FALSE""",
    conn,
)
conn.close()
print(check.to_string(index=False))
assert check['con_comp_ratio'][0] == 150, 'UPDATE incompleto'
print('\n[OK] Step 02 completo: 150 casos con 16 features en parquet y PostgreSQL.')

 total  con_comp_ratio  con_paradoja
   150             150           150

[OK] Step 02 completo: 150 casos con 16 features en parquet y PostgreSQL.


/var/folders/fp/7jqy71qx7tg6j0vg_jwpvqrc0000gn/T/ipykernel_73674/2428732984.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  check = pd.read_sql(
